# 303: Missing-member aggregation and local backfill

A small, scenario/member-dependent fraction of MAGICC probabilistic ensemble members
can fail with a native `NaN forcing found` fatal error on the cluster's Linux build for
reasons still under investigation (see `notes/` and the upstream GitLab issue) - this
appears to be a compiler/runtime-build sensitivity, not something our own code can
prevent. `301_run_magicc.ipynb` now handles this gracefully (proceeds with whichever
members survive rather than crashing the whole scenario) and tracks exactly which
`run_id`s are missing per `(model, scenario, flag)` under
`OUTPUT_FOLDER/missing_members/`.

This notebook has two parts, run in two different places:

- **Part A - run on the cluster**, after the full production batch has finished.
  Aggregates all the small per-scenario tracking files into one consolidated manifest
  small enough to move to a local machine directly (e.g. via a shared drive).
- **Part B - run locally** (Mac builds have shown 0% failures for every scenario
  tested so far - see `notes/`), once the manifest and `OUTPUT_FOLDER` are both
  reachable locally. Reruns *only* the specific missing members (not whole scenarios)
  and appends the recovered rows into the existing per-scenario/flag CSVs in place.

In [ ]:
import os
from pathlib import Path

import dotenv
import pandas as pd

dotenv.load_dotenv()
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])

## Part A - aggregate missing-member tracking files (run on the cluster)

Run this section once the full production batch (all shards) has completed. Skip
straight to Part B if you're running this notebook locally with an already-aggregated
manifest in hand.

In [ ]:
missing_members_dir = OUTPUT_FOLDER / "missing_members"
tracking_files = sorted(missing_members_dir.glob("*.csv"))
print(f"Found {len(tracking_files)} tracking file(s) under {missing_members_dir}")

In [ ]:
if tracking_files:
    missing_manifest = pd.concat(
        [pd.read_csv(f) for f in tracking_files], ignore_index=True
    )
else:
    missing_manifest = pd.DataFrame(columns=["model", "scenario", "flag", "run_id"])

n_scenarios_affected = missing_manifest[["model", "scenario"]].drop_duplicates().shape[0]
print(f"Total missing (model, scenario, flag, run_id) rows: {len(missing_manifest)}")
print(f"Scenarios with at least one missing member: {n_scenarios_affected}")
missing_manifest.groupby("flag").size().rename("missing_rows")

In [ ]:
manifest_path = OUTPUT_FOLDER / "missing_members_manifest.csv"
missing_manifest.to_csv(manifest_path, index=False)
print(f"Saved consolidated manifest to: {manifest_path}")
print("This single file is what needs to move to the local machine for Part B - "
      "the individual per-scenario tracking files under missing_members/ don't need "
      "to be copied separately.")

## Part B - local backfill rerun and merge

Run this section locally, once `missing_members_manifest.csv` (from Part A) and
`OUTPUT_FOLDER` (containing `individual_runs_raw/` and the `300_all_batches_v0.6.csv`
batch file) are both reachable from this machine - e.g. both on a shared drive, or
copied over.

For each affected `(model, scenario)`, this reruns MAGICC for the *union* of missing
`run_id`s across all three flags in one pass (cheap - a handful of members, not 600),
using the exact same harvest + QEXTRA logic as `301_run_magicc.ipynb`, then merges the
recovered **raw** rows into the existing raw output CSVs in place.

Deliberately no calibration/post-processing happens here - `CMIP7ScenarioMIPPostProcessor`'s
calibration shift is a single scalar computed from the *median across whichever
ensemble members are present* in the call that computes it, so post-processing this
small backfilled subset on its own (as an earlier version of this notebook did) would
bake in a spurious, systematic offset relative to the main batch's own calibration -
see `notes/` for the full trace. Calibration is applied exactly once, on the complete
merged raw ensemble, by `304_post_process_gsat.ipynb`, run after this section.

In [ ]:
import multiprocessing as _mp
try:
    _mp.set_start_method("fork")
except RuntimeError:
    pass

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("pint.util").setLevel(logging.ERROR)

import platform

import pandas_indexing as pix
from pandas_openscm.io import load_timeseries_csv

from gcages.cmip7_scenariomip.scm_running import CMIP7ScenarioMIPSCMRunner

from utils import sanitize_label, write_magicc_extra_rf_file

In [ ]:
N_PROCESSES = 8  # local backfill re-runs are small (a handful of members per scenario)

BASE_DIR = Path("../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs")
CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS = BASE_DIR / "history_cmip7_scenariomip.csv"
MAGICC_EXE_PATH = BASE_DIR / "magicc-v7.6.0a3" / "bin"
MAGICC_CMIP7_SCENARIOMIP_PROBABILISTIC_CONFIG_FILE = (
    BASE_DIR / "magicc-v7.6.0a3" / "configs" / "magicc-ar7-fast-track-drawnset-v0-3-0.json"
)

if platform.system() == "Darwin" and platform.processor() == "arm":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc-darwin-arm64"
    os.environ["DYLD_LIBRARY_PATH"] = "/opt/homebrew/Cellar/gcc/15.2.0_1/lib/gcc/current"
elif platform.system() == "Linux":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc"
elif platform.system() == "Windows":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc.exe"
else:
    raise RuntimeError(f"Unhandled platform: {platform.system()}")

print(f"Using MAGICC_EXE: {MAGICC_EXE}")

In [ ]:
CO2_EFFRF_COMPONENTS = (
    "Effective Radiative Forcing|CO2",
)

KYOTO_EFFRF_COMPONENTS = (
    "Effective Radiative Forcing|CO2",
    "Effective Radiative Forcing|CH4",
    "Effective Radiative Forcing|N2O",
    "Effective Radiative Forcing|F-Gases",
)

QEXTRA_BASKETS = {
    "CO2": CO2_EFFRF_COMPONENTS,
    "GHG": KYOTO_EFFRF_COMPONENTS,
}

BASKET_DESCRIPTION = {
    "CO2": "CO2-only",
    "GHG": "Kyoto-gases-only (CO2+CH4+N2O+F-Gases, no Montreal halogens)",
}


def build_scm_runner_restricted(output_variables, run_ids):
    """Build a scm runner restricted to exactly the given run_ids (paraset_id values,
    an intrinsic per-member identifier - see 301_run_magicc.ipynb's notes on this).
    Order is sorted for determinism; this is a small subset, not the full 600."""
    runner = CMIP7ScenarioMIPSCMRunner.from_cmip7_scenariomip_config(
        n_processes=N_PROCESSES,
        magicc_exe_path=MAGICC_EXE,
        magicc_prob_distribution_path=MAGICC_CMIP7_SCENARIOMIP_PROBABILISTIC_CONFIG_FILE,
        historical_emissions_path=CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS,
        output_variables=output_variables,
        batch_size_scenarios=15,
    )
    cfgs_by_id = {c["run_id"]: c for c in runner.climate_models_cfgs["MAGICC7"]}
    runner.climate_models_cfgs["MAGICC7"] = [cfgs_by_id[rid] for rid in sorted(run_ids)]
    return runner


def sum_effrf_components(erf_results, components):
    summed = (
        erf_results
        .loc[pix.isin(variable=list(components))]
        .groupby(level="run_id")
        .sum(numeric_only=True)
    )
    year_columns = [c for c in summed.columns if isinstance(c, int)]
    return summed[year_columns]

In [ ]:
missing_manifest = pd.read_csv(OUTPUT_FOLDER / "missing_members_manifest.csv")
batch_df = pd.read_csv(OUTPUT_FOLDER / "300_all_batches_v0.6.csv", index_col=0)
batch_lookup = batch_df.set_index(["MODEL", "SCENARIO"])

print(f"{len(missing_manifest)} missing rows across "
      f"{missing_manifest[['model', 'scenario']].drop_duplicates().shape[0]} scenarios")

In [ ]:
output_dir = OUTPUT_FOLDER / "individual_runs_raw"

backfill_summary = []

for (model, scenario), group in missing_manifest.groupby(["model", "scenario"]):
    sanitized_model = sanitize_label(model)
    sanitized_scenario = sanitize_label(scenario)

    file_row = batch_lookup.loc[(model, scenario)]
    scenario_file = Path(file_row["FILE"])

    # Union of missing run_ids across whichever flags need them for this scenario -
    # one harvest pass covers all of them at once.
    union_run_ids = sorted(group["run_id"].unique())
    missing_by_flag = {
        flag: set(sub) for flag, sub in group.groupby("flag")["run_id"]
    }

    print(f"\n=== {model} | {scenario} - backfilling {len(union_run_ids)} member(s), "
          f"needed for flags: {sorted(missing_by_flag)} ===")

    start = load_timeseries_csv(
        scenario_file,
        index_columns=["model", "scenario", "region", "variable", "unit"],
        out_columns_type=int,
        out_columns_name="year",
    )
    scenario_to_run = start.loc[pd.IndexSlice[model, scenario, :, :, :], :]

    # Harvest pass, restricted to the union of missing members.
    erf_runner = build_scm_runner_restricted(
        output_variables=("Surface Air Temperature Change", *KYOTO_EFFRF_COMPONENTS),
        run_ids=union_run_ids,
    )
    erf_runner.climate_models_cfgs["MAGICC7"] = [
        {**c, "rf_total_runmodus": "ANTHROPOGENIC"}
        for c in erf_runner.climate_models_cfgs["MAGICC7"]
    ]
    erf_results = erf_runner(scenario_to_run)
    anthropogenic_gsat = erf_results.loc[pix.isin(variable=["Surface Air Temperature Change"])]

    survived_after_harvest = set(anthropogenic_gsat.index.get_level_values("run_id"))
    still_missing = set(union_run_ids) - survived_after_harvest
    if still_missing:
        print(f"  WARNING: {len(still_missing)} member(s) failed again even locally: "
              f"{sorted(still_missing)} - will remain missing, needs manual follow-up.")

    qextra_effrf_by_flag = {
        flag: sum_effrf_components(erf_results, components)
        for flag, components in QEXTRA_BASKETS.items()
    }

    results_by_flag = {"ANTHROPOGENIC": anthropogenic_gsat}
    for flag, qextra_effrf in qextra_effrf_by_flag.items():
        qextra_forcing_dir = (
            OUTPUT_FOLDER / "qextra_forcing" / f"{sanitized_model}_{sanitized_scenario}_{flag}_backfill"
        )
        for run_id, row in qextra_effrf.iterrows():
            write_magicc_extra_rf_file(
                qextra_forcing_dir / f"member_{run_id}.IN",
                years=list(row.index),
                values=list(row.values),
                description=(
                    f"{BASKET_DESCRIPTION[flag]} ERF backfill: {model} | {scenario}, "
                    f"ensemble member {run_id}"
                ),
            )
        qextra_runner = build_scm_runner_restricted(
            output_variables=("Surface Air Temperature Change",),
            run_ids=list(qextra_effrf.index),
        )
        qextra_runner.climate_models_cfgs["MAGICC7"] = [
            {
                **c,
                "file_extra_rf": str(qextra_forcing_dir / f"member_{c['run_id']}.IN"),
                "rf_extra_read": 1,
                "rf_total_runmodus": "QEXTRA",
            }
            for c in qextra_runner.climate_models_cfgs["MAGICC7"]
        ]
        results_by_flag[flag] = qextra_runner(scenario_to_run)

    for flag, gsat_results in results_by_flag.items():
        if flag not in missing_by_flag:
            continue  # this scenario didn't actually need a backfill for this flag

        # Raw (pre-calibration) rows - see the note at the top of Part B on why
        # post-processing does not happen here.
        new_rows = gsat_results.loc[pix.isin(variable=["Surface Air Temperature Change"])]
        # Only keep the run_ids actually recorded as missing for this specific flag -
        # the union pass may have recovered a few extra members another flag needed.
        new_rows = new_rows.loc[
            new_rows.index.get_level_values("run_id").isin(missing_by_flag[flag])
        ]

        output_path = output_dir / f"{sanitized_model}_{sanitized_scenario}_{flag}.csv"
        existing = pd.read_csv(output_path)
        existing = existing.set_index(
            [c for c in existing.columns if not c.isdigit()]
        )
        existing.columns = existing.columns.astype(int)

        combined = pd.concat([existing, new_rows])
        combined = combined[~combined.index.get_level_values("run_id").duplicated(keep="first")]
        combined = combined.sort_index(level="run_id")
        combined.to_csv(output_path)

        n_recovered = len(new_rows)
        n_still_missing = len(missing_by_flag[flag]) - n_recovered
        print(f"  [{flag}] recovered {n_recovered}/{len(missing_by_flag[flag])} member(s) "
              f"-> {output_path}" + (f" ({n_still_missing} still missing)" if n_still_missing else ""))

        backfill_summary.append({
            "model": model, "scenario": scenario, "flag": flag,
            "needed": len(missing_by_flag[flag]), "recovered": n_recovered,
        })

In [ ]:
summary_df = pd.DataFrame(backfill_summary, columns=["model", "scenario", "flag", "needed", "recovered"])
if summary_df.empty:
    print("Nothing was backfilled (missing_members_manifest.csv was empty).")
else:
    print(f"\nBackfill complete: {summary_df['recovered'].sum()}/{summary_df['needed'].sum()} "
          f"member(s) recovered across {len(summary_df)} (scenario, flag) group(s).")
summary_df[summary_df["recovered"] < summary_df["needed"]]